# 9장. 사전학습: 이야기를 읽으며 배우기

8장의 빠른 GPT를 이야기 20만 편으로 학습해요. **T4 GPU가 필요하고, 30분 정도 걸립니다.** (런타임 유형: T4 GPU)

**이 장이 끝나면**: 드라이브에 `pretrain.pt`(학습된 모델)와 단계별 체크포인트가 저장됩니다.
GPU를 받지 못했거나 시간이 없다면 이 장은 읽기만 해도 돼요. 10장은 드라이브에 `pretrain.pt`가 없으면 저자의 체크포인트를 내려받습니다.


### 준비
드라이브 연결 창이 뜨면 허용하세요.


In [ ]:
%pip install -q git+https://github.com/yhlee1627/modu-llm

import os
import torch
from modu_llm import Vocab, GPTConfig, GPT, count_params, generate, get_batch
from modu_llm import prepare_stories, load_checkpoint, train, mount_drive
from modu_llm import get_device, plot_history

device = get_device(need_gpu=True)
vocab = Vocab.build()
folder = mount_drive()


In [ ]:
# 학습 설정
N_STORIES = 200_000      # 학습에 쓸 이야기 수
TRAIN_MINUTES = 30       # 이번 실행에서 학습할 시간(분)
LR = 5e-4                # 학습률: 한 번에 가중치를 고치는 정도
BATCH = 32               # 한 단계에 꺼낼 조각 수
EVAL_EVERY = 250         # 몇 단계마다 손실을 확인할지
SNAPSHOTS = [250, 1000, 3000]   # 10장에서 비교할 단계별 체크포인트
ckpt_path = f"{folder}/pretrain.pt"


## 9.1 학습 데이터 준비: 메모리를 아끼는 인코딩

이야기 20만 편은 약 7천만 글자예요. 2장처럼 한 번에 번호 리스트로 바꾸면 파이썬 정수 수천만 개가 생겨 Colab 메모리가 부족해질 수 있습니다.
그래서 패키지의 `prepare_stories`가 **이야기마다** 인코딩해 작은 정수 배열(`uint16`, 숫자 하나에 2바이트)로 이어 붙여요. 한 번 만든 배열은 드라이브에 저장해 두고 다음부터는 불러오기만 합니다.

**핵심 코드**: `train_data, val_data = prepare_stories(N_STORIES, vocab, folder)`


In [ ]:
train_data, val_data = prepare_stories(N_STORIES, vocab, folder)


## 9.2 학습 한 단계: 예측하고, 틀린 만큼 고치기

먼저 파라미터가 **w 하나뿐인** 작은 문제로 "고치기"가 어떤 계산인지 손으로 확인해요. 정답은 w = 2(받은 숫자의 2배)입니다.
`loss.backward()`가 기울기를 `w.grad`에 넣어 주면, **기울기의 반대쪽으로 학습률만큼** w를 고칩니다. 이것이 경사 하강이에요.

**핵심 코드**: `w -= lr * w.grad`


In [ ]:
x = torch.tensor([1.0, 2.0, 3.0])          # 문제
y = 2 * x                                  # 정답: 문제의 2배
w = torch.tensor(0.0, requires_grad=True)  # 고칠 숫자 하나, 0에서 시작
lr = 0.05                                  # 학습률

for i in range(5):
    loss = ((w * x - y) ** 2).mean()       # ① 예측과 손실
    loss.backward()                        # ③ 고칠 방향 계산
    print(f"단계 {i + 1}: w {w.item():.3f} | 손실 {loss.item():.3f} | 기울기 {w.grad.item():.3f}")
    with torch.no_grad():
        w -= lr * w.grad                   # ④ 기울기 반대쪽으로 lr만큼 고치기
    w.grad.zero_()                         # ② 다음 단계를 위해 기울기 지우기
print(f"5단계 뒤: w {w.item():.3f}")


고치는 두 줄을 옵티마이저 `torch.optim.SGD`에게 맡기고, 학습률을 바꿔 봐요. 너무 작으면 느리고, 너무 크면 **발산**합니다.
리스트에 0.2를 넣어 보면 2 근처를 오가며 다가가는 모습도 볼 수 있어요.


In [ ]:
def fit(lr, steps=5):
    w = torch.tensor(0.0, requires_grad=True)
    optimizer = torch.optim.SGD([w], lr=lr)
    for _ in range(steps):
        loss = ((w * x - y) ** 2).mean()   # ① 예측과 손실
        optimizer.zero_grad()              # ② 지난 기울기 지우기
        loss.backward()                    # ③ 고칠 방향 계산
        optimizer.step()                   # ④ 가중치 고치기
    return w.item()

for lr in [0.01, 0.05, 0.25]:
    print(f"학습률 {lr}: 5단계 뒤 w = {fit(lr):.3f}")


이제 같은 네 줄을 609만 개짜리 GPT에 씁니다. 옵티마이저는 SGD에 관성과 숫자마다 다른 보폭을 더한 **AdamW**예요.

1. 배치를 꺼내 **예측**하고 손실을 계산한다 (`model(x, y)`)
2. 지난 기울기를 지운다 (`zero_grad`)
3. 손실을 줄이려면 가중치를 어느 방향으로 고칠지 계산한다 (`backward`)
4. 실제로 고친다 (`step`)

아래 셀은 다섯 단계만 실행해 손실이 줄어드는지 확인해요.


In [ ]:
model = GPT(GPTConfig()).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

def train_step():
    x, y = get_batch(train_data, model.cfg.block_size, BATCH, device)
    _, loss = model(x, y)                  # ① 예측과 손실
    optimizer.zero_grad()                  # ② 지난 기울기 지우기
    loss.backward()                        # ③ 고칠 방향 계산
    optimizer.step()                       # ④ 가중치 고치기
    return loss.item()

n_params = count_params(model)
print(f"파라미터 수: {n_params:,}")
for i in range(5):
    loss = train_step()
    print(f"단계 {i + 1}: 손실 {loss:.3f}")


### 검증 손실 재기
학습에 쓰지 않은 `val_data`로 손실을 재요. 고치지 않으므로 `no_grad` 안에서 재고, 다섯 배치의 평균을 냅니다.
9.3의 `train`도 `EVAL_EVERY`단계마다 같은 일을 해요.


In [ ]:
model.eval()                               # 시험 모드
with torch.no_grad():                      # 재기만 하고 고치지 않음
    val_losses = []
    for _ in range(5):
        x, y = get_batch(val_data, model.cfg.block_size, BATCH, device)
        _, loss = model(x, y)
        val_losses.append(loss.item())
model.train()                              # 다시 공부 모드
print(f"검증 손실(5배치 평균): {sum(val_losses) / len(val_losses):.3f}")


## 9.3 끊겨도 이어서 학습하기

본 학습입니다. 드라이브에 `pretrain.pt`가 있으면 **그 상태에서 이어서**, 없으면 처음부터 학습해요.
학습은 패키지의 `train` 함수가 맡습니다. 9.2의 학습 한 단계를 `TRAIN_MINUTES`분 동안 되풀이하면서, 손실을 기록하고 5분마다 **체크포인트**(학습 중간 상태 파일)를 저장해요.
세션이 끊기면 런타임을 다시 연결하고 **준비 셀부터 다시 실행**하세요. 저장된 지점부터 이어집니다.

**핵심 코드**: `if os.path.exists(ckpt_path): ... load_checkpoint(...)`


In [ ]:
if os.path.exists(ckpt_path):
    model, optimizer, step, history = load_checkpoint(ckpt_path, device, optimizer_lr=LR)
    print(f"{step}단계에서 이어서 학습합니다.")
else:
    model = GPT(GPTConfig()).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    step = 0
    history = []
    print("처음부터 학습합니다.")

step, history = train(
    model, optimizer, train_data, val_data,
    minutes=TRAIN_MINUTES,
    ckpt_path=ckpt_path,
    eval_every=EVAL_EVERY,
    start_step=step,
    history=history,
    snapshots=SNAPSHOTS,
    vocab=vocab,
)

last = history[-1]
print(f"학습 끝: {step}단계 | 검증 손실 {last['val']:.3f} | 저장: {ckpt_path}")


## 9.4 손실 그래프 읽기와 과적합

학습 손실과 검증 손실이 **함께** 내려가면 처음 보는 이야기에도 잘 통하도록 배우고 있다는 뜻이에요.
학습 손실만 내려가고 검증 손실이 올라가기 시작하면 **과적합**, 즉 학습한 글을 외우기 시작했다는 신호입니다.


In [ ]:
plot_history(history)


## 놀이터

본 모델을 건드리지 않는 활동입니다. 건너뛰어도 다음 장에 영향이 없습니다.

여러 시작 문장으로 이야기를 만들어 보세요. 1장의 2분짜리 모델과 비교하면 어떤가요?


In [ ]:
for prompt in ["어느 날, ", "밤하늘에 ", "엄마가 "]:
    story = generate(model, vocab, prompt, max_new_tokens=200)
    print(prompt + story)
    print("-" * 40)


## 실제 LLM에서는

- 실제 LLM의 사전학습은 수천 개의 GPU로 수조 개의 토큰을 몇 주에서 몇 달 동안 학습합니다.
- 학습률을 처음에 천천히 올렸다가 점점 낮추는 일정(warmup, cosine)을 쓰고, 웹 문서·책·코드 등 여러 데이터를 비율을 정해 섞어요.
